In [1]:
%load_ext autoreload
%autoreload 2

## 1. Imports

In [2]:
import os
import sys

sys.path.append("..")

import random

import numpy as np
import torch
import torch.distributions as TD
import wandb
from tqdm import tqdm

from src.costs.lse import MLPLSECost
from src.models.energy_based import EGEOT
from src.plotting.distributions import plot_swiss_roll
from src.plotting.parameters import plot_B_parameters
from src.potentials.mlp_based import MLPPotential
from src.samplers.energy_based.sample_buffer import SampleBufferEgEOT
from src.samplers.from_dataset import DatasetSampler
from src.samplers.primary import StandardNormalSampler, SwissRollSampler
from src.utils.discrete_ot import OTPlanSampler
from src.utils.paired import generate_paired_data, get_GT_points, get_paired_sampler
from src.utils.train import compute_loss, update_average

In [3]:
device = torch.device(f"cuda:{torch.cuda.current_device()}" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [4]:
torch.set_default_device(device)
dtype = torch.float64
torch.torch.set_default_dtype(dtype)

## 2. Config

In [5]:
# TODO: add binding between config and class instance
from configs.energy_based.cost import CostConfig
from configs.energy_based.dataset import DatasetConfig, MiniBatchConfig
from configs.energy_based.model import EBMConfig
from configs.energy_based.optimizer import OptPairedConfig, OptUnpairedConfig
from configs.energy_based.potential import PotentialConfig
from configs.energy_based.train import TrainConfig

In [6]:
dataset_config = DatasetConfig()
minibatch_config = MiniBatchConfig()

potential_config = PotentialConfig()
cost_config = CostConfig()
model_config = EBMConfig()

opt_unpaired_config = OptUnpairedConfig()
opt_paired_config = OptPairedConfig()

train_config = TrainConfig()

In [7]:
torch.manual_seed(train_config.seed)
np.random.seed(train_config.seed)
random.seed(train_config.seed)

## 3. Create data and samplers

In [8]:
X_sampler = StandardNormalSampler(dim=dataset_config.x_dim, device=device)
Y_sampler = SwissRollSampler(dim=dataset_config.y_dim, device=device, dtype=dtype)

In [9]:
otp_sampler = OTPlanSampler(**minibatch_config.model_dump())

In [10]:
data_dir = "checkpoints/Tensors"
file_postfix = f"{minibatch_config.cost_function}_{dataset_config.P_XY_paired}"

In [11]:
X_paired_train, Y_paired_train, X_paired_test, Y_paired_test = generate_paired_data(
    X_sampler, Y_sampler, otp_sampler, dataset_config.P_XY_paired, "./checkpoints/Tensors", file_postfix, device=device
)

In [12]:
pd_train_sampler = get_paired_sampler(
    X_paired_train, Y_paired_train, train_config.batch_size, dataset_config.P_XY_paired, device
)

In [13]:
X_unpaired_test = X_sampler.sample(dataset_config.P_XY_paired)
Y_unpaired_test = Y_sampler.sample(dataset_config.P_XY_paired)

In [14]:
if dataset_config.Q_X_unpaired > 0:
    source_data = X_sampler.sample(dataset_config.Q_X_unpaired )
    usd_sampler = DatasetSampler(source_data, device=device) # usd - unpaired source data
else:
    usd_sampler = DatasetSampler(X_paired_train, device=device)

if dataset_config.R_Y_unpaired > 0:
    target_data = Y_sampler.sample(dataset_config.R_Y_unpaired)
    utd_sampler = DatasetSampler(target_data, device=device) # utd - unpaired target data
else:
    utd_sampler = DatasetSampler(Y_paired_train, device=device)

## 4. Model initialization

In [16]:
potential = MLPPotential(**potential_config.model_dump())

In [17]:
cost = MLPLSECost(**cost_config.model_dump())

In [18]:
# TODO: add to config
BASIC_NOISE_VAR = 1.0
P_SAMPLE_BUFFER_REPLAY = 0.95
SAMPLE_BUFFER_SAMPLES = 10000

In [19]:
basic_noise_gen = TD.Normal(torch.tensor([0.0, 0.0]).to(device), torch.tensor([1.0, 1.0]).to(device) * BASIC_NOISE_VAR)

sample_buffer_instance = SampleBufferEgEOT(
    basic_noise_gen, p=P_SAMPLE_BUFFER_REPLAY, max_samples=SAMPLE_BUFFER_SAMPLES, device=device
)

In [20]:
model = EGEOT(potential, cost, sample_buffer_instance, model_config)

In [21]:
# For EMA update
if train_config.ema_update:
    model_copy = EGEOT(potential, cost, sample_buffer_instance, model_config)

## 5. Optimizers initialization

In [22]:
unpaired_params_to_update = model.potential.parameters()

D_opt_unpaired = torch.optim.Adam(unpaired_params_to_update, **opt_unpaired_config.model_dump())

In [23]:
paired_params_to_update = model.cost.parameters()

D_opt_paired = torch.optim.Adam(paired_params_to_update, **opt_paired_config.model_dump())

In [24]:
# TODO: refactor this config
EXP_META_INFO = ""
EXP_NAME = (
    "EgEOT_Swiss_Roll_"
    + f"P_XY_PAIRED_{dataset_config.P_XY_paired}_"
    + f"Q_X_UNPAIRED_{dataset_config.Q_X_unpaired}_"
    + f"R_Y_UNPAIRED_{dataset_config.R_Y_unpaired}_"
    + f"LR_PAIRED_{opt_paired_config.lr}_"
    + f"LR_UNPAIRED_{opt_unpaired_config.lr}_"
    + f"MINIBATCH_COST_{minibatch_config.cost_function}_"
    + EXP_META_INFO
)
OUTPUT_PATH = "../checkpoints/{}".format(EXP_NAME)

config = dict(
    X_DIM=dataset_config.x_dim,
    Y_DIM=dataset_config.y_dim,
    D_LR_PAIRED=opt_paired_config.lr,
    D_LR_UNPAIRED=opt_unpaired_config.lr,
    BATCH_SIZE=train_config.batch_size,
    P_XY_PAIRED_SAMPLES=dataset_config.P_XY_paired,
    Q_X_UNPAIRED_SAMPLES=dataset_config.Q_X_unpaired,
    R_Y_UNPAIRED_SAMPLES=dataset_config.R_Y_unpaired,
)

if not os.path.exists(OUTPUT_PATH):
    os.makedirs(OUTPUT_PATH)

In [25]:
if train_config.steps_from > 0:
    D_opt_unpaired.load_state_dict(torch.load(os.path.join(OUTPUT_PATH, f"D_opt_unpaired_{train_config.steps_from}.pt")))
    D_opt_paired.load_state_dict(torch.load(os.path.join(OUTPUT_PATH, f"D_opt_paired_{train_config.steps_from}.pt")))

## 6. Model training

In [26]:
starting_points = torch.tensor([[-2.0, 0.0], [0.0, 0.0], [0.0, -2.0]])
num_ending_points = 64

In [27]:
num_starting_points_paired = 5
indices = random.choices(range(dataset_config.P_XY_paired), k=num_starting_points_paired)
starting_points_paired = X_paired_train[indices]
ending_points_paired = Y_paired_train[indices]

In [28]:
gt_Y_points = get_GT_points(X_sampler, Y_sampler, otp_sampler, starting_points)

  0%|          | 0/64 [00:00<?, ?it/s]/Users/michael/miniconda3/envs/light-gcot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:531: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn("Sinkhorn did not converge. You might want to "
100%|██████████| 64/64 [00:00<00:00, 185.62it/s]


In [52]:
wandb.init(name=EXP_NAME, config=config)

for step in tqdm(range(train_config.steps_from, train_config.steps_to)):
    # training loop
    D_opt_unpaired.zero_grad()

    X = usd_sampler.sample(train_config.batch_size)
    Y = utd_sampler.sample(train_config.batch_size)

    output = model.compute_unpaired_loss(X, Y)

    D_loss_unpaired = output["loss"]
    wandb.log({f"D unpaired loss": D_loss_unpaired.item()}, step=step)

    wandb.log({f"Pos Out Mean": output["pos_out"].item()}, step=step)
    wandb.log({f"Neg Out Mean": output["neg_out"].item()}, step=step)
    wandb.log({f"D unpaired loss": D_loss_unpaired.item()}, step=step)
    wandb.log({f"r_t": output["r_t"].item()}, step=step)
    wandb.log({f"cost_r_t": output["cost_r_t"].item()}, step=step)
    wandb.log({f"score_r_t": output["score_r_t"].item()}, step=step)
    wandb.log({f"noise": output["noise"].item()}, step=step)

    D_opt_paired.zero_grad()
    X_paired, Y_paired = pd_train_sampler.sample(train_config.batch_size)

    D_loss_paired = model.compute_paired_loss(X_paired, Y_paired)
    wandb.log({f"D paired loss": D_loss_paired.item()}, step=step)

    D_loss = D_loss_unpaired + D_loss_paired
    D_loss.backward()
    D_opt_paired.step()
    D_opt_unpaired.step()

    D_unpaired_gradient_norm = torch.nn.utils.clip_grad_norm_(
        unpaired_params_to_update, max_norm=train_config.gradient_max_norm
    )
    wandb.log({f"D unpaired gradient norm": D_unpaired_gradient_norm.item()}, step=step)
    D_paired_gradient_norm = torch.nn.utils.clip_grad_norm_(
        paired_params_to_update, max_norm=train_config.gradient_max_norm
    )
    wandb.log({f"D paired gradient norm": D_paired_gradient_norm.item()}, step=step)

    if train_config.ema_update:
        update_average(model_copy, model, 0.99)
        model = model_copy
    else:
        model = model

    wandb.log({f"D loss": D_loss}, step=step)
    wandb.log(
        {f"Train paired loss": compute_loss(model, X_paired_train, Y_paired_train, X_paired_train, Y_paired_train)},
        step=step,
    )
    wandb.log(
        {f"Test paired loss": compute_loss(model, X_paired_test, Y_paired_test, X_paired_test, Y_paired_test)},
        step=step,
    )
    wandb.log(
        {f"Test unpaired loss": compute_loss(model, X_unpaired_test, Y_unpaired_test, X_paired_test, Y_paired_test)},
        step=step,
    )

    if step % train_config.plot_every == 0:
        B_dict = plot_B_parameters(model.cost, starting_points, log=True)
        distr_dict = plot_swiss_roll(
            {"EBM": model},
            X_sampler,
            Y_sampler,
            X_paired_train,
            Y_paired_train,
            starting_points,
            gt_Y_points,
            log=True,
        )
        wandb.log(distr_dict | B_dict)
        torch.save(model.potential.state_dict(), os.path.join(OUTPUT_PATH, f"potential_{step}.pt"))

torch.save(model.potential.state_dict(), os.path.join(OUTPUT_PATH, f"D_{train_config.steps_to}.pt"))
torch.save(D_opt_paired.state_dict(), os.path.join(OUTPUT_PATH, f"D_opt_paired_{train_config.steps_to}.pt"))
torch.save(D_opt_unpaired.state_dict(), os.path.join(OUTPUT_PATH, f"D_opt_unpaired_{train_config.steps_to}.pt"))

wandb.finish()

  1%|          | 6/1000 [20:27<56:27:53, 204.50s/it]


KeyboardInterrupt: 

## Plotting

In [ ]:
plot_swiss_roll(
    {"EBM": model},
    X_sampler,
    Y_sampler,
    X_paired_train,
    Y_paired_train,
    starting_points,
    gt_Y_points,
) 